# Garim 장면분할 단독 테스트 Colab

이 노트북은 **PySceneDetect 기반 장면분할만** 검증합니다.

지원 입력:
- `YOUTUBE_URL` 값이 있으면 YouTube 영상 다운로드 후 장면분할
- `YOUTUBE_URL` 값이 비어 있으면 로컬 영상 업로드 후 장면분할

출력:
- 영상 기본 정보
- 장면분할 결과 CSV
- 장면별 대표 프레임 이미지
- 장면분할 결과 ZIP

In [ ]:
# 1. 라이브러리 설치
!pip install -q "scenedetect[opencv]" yt-dlp pandas matplotlib

In [ ]:
# 2. 설정값

from pathlib import Path

# 유튜브 링크가 있으면 입력하세요.
# 비워두면 파일 업로드 방식으로 진행됩니다.
YOUTUBE_URL = ""

# 장면분할 민감도
# 값이 낮을수록 장면이 더 많이 나뉩니다.
# 추천:
# - 일반 영상: 18.0
# - 장면 전환이 잘 안 잡히면: 12.0 ~ 15.0
# - 너무 많이 나뉘면: 25.0 ~ 30.0
THRESHOLD = 18.0

WORK_DIR = Path("/content/garim_scene_split_test")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
THUMB_DIR = OUTPUT_DIR / "scene_thumbnails"

for d in [INPUT_DIR, OUTPUT_DIR, THUMB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("작업 폴더:", WORK_DIR)
print("THRESHOLD:", THRESHOLD)

In [ ]:
# 3. 영상 입력
# - YOUTUBE_URL이 있으면 yt-dlp로 다운로드
# - 없으면 직접 업로드

from google.colab import files
import subprocess
import shutil
import os

def prepare_input_video(youtube_url: str) -> Path:
    youtube_url = youtube_url.strip()

    if youtube_url:
        video_path = INPUT_DIR / "youtube_input.mp4"
        print("YouTube 영상 다운로드 중...")
        cmd = [
            "yt-dlp",
            "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/best",
            "--merge-output-format", "mp4",
            "-o", str(video_path),
            youtube_url,
        ]
        subprocess.run(cmd, check=True)
        print("다운로드 완료:", video_path)
        return video_path

    print("영상 파일을 업로드하세요. 예: mp4, mov, avi, mkv, webm")
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("업로드된 파일이 없습니다.")

    uploaded_name = next(iter(uploaded.keys()))
    src_path = Path(uploaded_name)
    dst_path = INPUT_DIR / uploaded_name

    if dst_path.exists():
        dst_path.unlink()

    shutil.move(str(src_path), str(dst_path))
    print("업로드 완료:", dst_path)
    return dst_path

video_path = prepare_input_video(YOUTUBE_URL)
print("분석 대상:", video_path)

In [ ]:
# 4. 영상 기본 정보 확인

import cv2
import json

def get_video_meta(path: Path) -> dict:
    cap = cv2.VideoCapture(str(path))

    if not cap.isOpened():
        raise RuntimeError(f"영상을 열 수 없습니다: {path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration_sec = frame_count / fps if fps else 0

    cap.release()

    return {
        "path": str(path),
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": duration_sec,
    }

META = get_video_meta(video_path)

print(json.dumps(META, ensure_ascii=False, indent=2))

In [ ]:
# 5. 프레임 읽기 정상 여부 확인
# 첫 프레임, 중간 프레임, 마지막 근처 프레임을 확인합니다.

import matplotlib.pyplot as plt
import cv2
import math

def read_frame(path: Path, frame_no: int):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_no)
    ret, frame = cap.read()
    cap.release()
    return ret, frame

check_frames = [
    0,
    max(0, META["frame_count"] // 2),
    max(0, META["frame_count"] - 5),
]

plt.figure(figsize=(15, 4))

for i, frame_no in enumerate(check_frames):
    ret, frame = read_frame(video_path, frame_no)

    plt.subplot(1, 3, i + 1)
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.imshow(frame_rgb)
        plt.title(f"Frame {frame_no}")
    else:
        plt.text(0.5, 0.5, f"Read failed\nFrame {frame_no}", ha="center", va="center")
        plt.title(f"Frame {frame_no}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 6. PySceneDetect 장면분할 실행

from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

def detect_scenes(video_path: Path, threshold: float):
    video = open_video(str(video_path))
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=threshold))

    print("장면분할 실행 중...")
    scene_manager.detect_scenes(video)

    scene_list = scene_manager.get_scene_list()
    return scene_list

scene_list = detect_scenes(video_path, THRESHOLD)

print("탐지된 장면 수:", len(scene_list))

In [ ]:
# 7. 장면분할 결과 DataFrame 생성

import pandas as pd

def build_scene_dataframe(scene_list, meta: dict) -> pd.DataFrame:
    rows = []

    for idx, (start, end) in enumerate(scene_list):
        start_sec = start.get_seconds()
        end_sec = end.get_seconds()

        rows.append({
            "scene_id": idx,
            "start_sec": round(start_sec, 3),
            "end_sec": round(end_sec, 3),
            "duration_sec": round(end_sec - start_sec, 3),
            "start_frame": start.get_frames(),
            "end_frame": end.get_frames(),
        })

    # 장면 전환이 하나도 안 잡힌 경우 전체 영상을 1개 장면으로 처리
    if not rows:
        rows.append({
            "scene_id": 0,
            "start_sec": 0.0,
            "end_sec": round(meta["duration_sec"], 3),
            "duration_sec": round(meta["duration_sec"], 3),
            "start_frame": 0,
            "end_frame": meta["frame_count"],
        })

    return pd.DataFrame(rows)

SCENES_DF = build_scene_dataframe(scene_list, META)
SCENES_DF

In [ ]:
# 8. CSV / JSON 저장

csv_path = OUTPUT_DIR / "scene_split_result.csv"
json_path = OUTPUT_DIR / "scene_split_result.json"
meta_path = OUTPUT_DIR / "video_meta.json"

SCENES_DF.to_csv(csv_path, index=False, encoding="utf-8-sig")
SCENES_DF.to_json(json_path, orient="records", force_ascii=False, indent=2)

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(META, f, ensure_ascii=False, indent=2)

print("CSV 저장:", csv_path)
print("JSON 저장:", json_path)
print("META 저장:", meta_path)

In [ ]:
# 9. 장면별 대표 프레임 저장
# 각 장면의 중간 프레임을 대표 이미지로 저장합니다.

import cv2

def save_scene_thumbnails(video_path: Path, scenes_df: pd.DataFrame, thumb_dir: Path):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError("영상을 열 수 없습니다.")

    saved_paths = []

    for _, row in scenes_df.iterrows():
        scene_id = int(row["scene_id"])
        start_frame = int(row["start_frame"])
        end_frame = int(row["end_frame"])

        mid_frame = max(0, (start_frame + end_frame) // 2)

        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
        ret, frame = cap.read()

        if not ret:
            print(f"[WARN] 대표 프레임 읽기 실패: scene={scene_id}, frame={mid_frame}")
            continue

        out_path = thumb_dir / f"scene_{scene_id:03d}_frame_{mid_frame}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved_paths.append(out_path)

    cap.release()
    return saved_paths

thumbnail_paths = save_scene_thumbnails(video_path, SCENES_DF, THUMB_DIR)

print("대표 프레임 저장 개수:", len(thumbnail_paths))
print("저장 위치:", THUMB_DIR)

In [ ]:
# 10. 대표 프레임 미리보기

import matplotlib.pyplot as plt
import cv2
import math

max_show = min(len(thumbnail_paths), 12)

if max_show == 0:
    print("표시할 대표 프레임이 없습니다.")
else:
    cols = 3
    rows = math.ceil(max_show / cols)

    plt.figure(figsize=(15, rows * 4))

    for i in range(max_show):
        img = cv2.imread(str(thumbnail_paths[i]))

        plt.subplot(rows, cols, i + 1)

        if img is None:
            plt.text(0.5, 0.5, "Image read failed", ha="center", va="center")
        else:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            scene_id = SCENES_DF.iloc[i]["scene_id"]
            start_sec = SCENES_DF.iloc[i]["start_sec"]
            end_sec = SCENES_DF.iloc[i]["end_sec"]

            plt.imshow(img_rgb)
            plt.title(f"Scene {scene_id}\n{start_sec}s ~ {end_sec}s")

        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# 11. 장면 경계 근처 프레임 확인
# 장면이 잘 나뉘었는지 확인하기 위해 각 장면의 시작 프레임을 저장합니다.

BOUNDARY_DIR = OUTPUT_DIR / "scene_boundaries"
BOUNDARY_DIR.mkdir(exist_ok=True)

def save_boundary_frames(video_path: Path, scenes_df: pd.DataFrame, out_dir: Path):
    cap = cv2.VideoCapture(str(video_path))
    saved = []

    for _, row in scenes_df.iterrows():
        scene_id = int(row["scene_id"])
        start_frame = int(row["start_frame"])

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        ret, frame = cap.read()

        if not ret:
            continue

        out_path = out_dir / f"scene_{scene_id:03d}_start_frame_{start_frame}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved.append(out_path)

    cap.release()
    return saved

boundary_paths = save_boundary_frames(video_path, SCENES_DF, BOUNDARY_DIR)

print("장면 시작 프레임 저장 개수:", len(boundary_paths))
print("저장 위치:", BOUNDARY_DIR)

In [ ]:
# 12. 결과 ZIP 생성 및 다운로드

import shutil
from google.colab import files

zip_base = Path("/content/garim_scene_split_result")
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)

print("ZIP 생성 완료:", zip_path)
files.download(zip_path)

In [ ]:
# 13. threshold 변경 재실행용 함수
# 장면이 너무 적거나 많으면 이 셀에서 threshold만 바꿔 빠르게 재실행하세요.

def rerun_scene_detection(new_threshold: float):
    global THRESHOLD, scene_list, SCENES_DF, thumbnail_paths

    THRESHOLD = new_threshold
    print("새 THRESHOLD:", THRESHOLD)

    scene_list = detect_scenes(video_path, THRESHOLD)
    print("탐지된 장면 수:", len(scene_list))

    SCENES_DF = build_scene_dataframe(scene_list, META)

    rerun_csv_path = OUTPUT_DIR / f"scene_split_result_threshold_{str(new_threshold).replace('.', '_')}.csv"
    SCENES_DF.to_csv(rerun_csv_path, index=False, encoding="utf-8-sig")

    print("CSV 저장:", rerun_csv_path)
    display(SCENES_DF.head(30))

    return SCENES_DF

# 예시:
# rerun_scene_detection(12.0)
# rerun_scene_detection(15.0)
# rerun_scene_detection(25.0)